# Day 5 — Strengthen Evidence, Failure Analysis & Efficiency

**Research:** Learning to Recover: Teaching Small Coding Agents to Fix Their Own Mistakes Using Execution Feedback

**Prior artifacts:** Day 2 manifest, Day 3 C1/C2 QLoRA adapters, Day 4 development outputs. Attach all three via **Kaggle Add Input** or private datasets containing their ZIP backups.

Today, we will:

1. **Reuse Day 4's 30 development tasks** instead of regenerating or rescoring them.
2. Add **80 new held-out MBPP+ development tasks** (chosen in advance), while reserving **70 further tasks untouched** for a possible final evaluation.
3. Extend the **fixed-failure comparison**, then evaluate **end-to-end 1/2/3 attempts** on an independently chosen subset.
4. Evaluate repeated mistakes, execution counts, generated tokens and **C2 without specific error feedback**.
5. Optionally test **single-shot HumanEval+ transfer**.
6. Use **offline official EvalPlus scoring**, paired bootstrap confidence intervals, figures and a downloaded ZIP backup.

**Safety:** The test runner uses subprocess limits, but *not a strong sandbox*. Run only on a disposable, credential-free system. Don't mount secrets or private files while evaluating arbitrary generated code. **Hidden tests are imported only in a separate scoring subprocess after all code generation finishes.**

**Research integrity:** This is an *extended development study*, not final untouched-test validation. Never tune adapters using final-holdout IDs. If results are inconclusive, report that honestly.

## A — Install dependencies (new Kaggle session)

In [ ]:
%pip install -q -U "transformers>=5.18.0" "peft>=0.19.0" accelerate datasets bitsandbytes safetensors tqdm evalplus matplotlib pandas

## B — Configuration and CUDA check

Run this before importing CUDA elsewhere. **Only one of your two T4 GPUs** is used for inference; the other cannot be pooled automatically. To adjust the computational budget, change the values below *before running any generation*. Once saved, task sets are frozen for reproducibility.

In [ ]:
import os
os.environ['CUDA_VISIBLE_DEVICES'] = '0'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'

import sys, json, torch
from pathlib import Path

print('Python:',sys.version.split()[0])
print('GPU:',torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NOT AVAILABLE')
assert torch.cuda.is_available(), 'Enable GPU T4 ×2 in Kaggle settings'
print('VRAM GiB:',round(torch.cuda.get_device_properties(0).total_memory/1024**3,2))

# Freeze these before running any generation; do not silently alter after seeing results.
NEW_DEV_TASKS = 80
FINAL_HOLDOUT_TASKS = 70
E2E_TASKS = 30
HUMANEVAL_TASKS = 20
MAX_NEW_TOKENS = 384
MAX_ATTEMPTS = 3
PARTITION_SEED = 20261109
E2E_SEED = 20261110
HUMANEVAL_SEED = 20261111

RUN_E2E = True
RUN_NO_FEEDBACK = True
RUN_HUMANEVAL = True

OUT = Path('/kaggle/working/day5_outputs')
OUT.mkdir(parents=True,exist_ok=True)
print('Output folder:',OUT)

## C — Write reusable evaluation modules from Day 4, plus Day 5 additions

In [ ]:
%%writefile /kaggle/working/test_runner.py
"""Small, resource-limited function-code runner for Kaggle Day 1.

SECURITY: subprocess + resource limits are NOT a strong sandbox. Model-generated
code can still access local files or network unless the external runtime forbids
it. Never run with secrets, private data or elevated access. For a public-facing
benchmark or adversarial model, run in a truly isolated container/service.
"""
from __future__ import annotations

import json
import os
import resource
import subprocess
import sys
import tempfile
from pathlib import Path

# Entire driver runs in a separate Python interpreter; no generated code is exec'd
# in the notebook kernel. Child stdout is reserved for a JSON response.
_CHILD = r'''
import ast, contextlib, io, json, sys, traceback
payload = json.load(sys.stdin)
source = payload["code"]
checks = payload["tests"]
setup = payload.get("setup", "") or ""

def trace(e):
    return "".join(traceback.format_exception_only(type(e), e)).strip()[:1800]

namespace = {"__name__": "__main__"}
result = {"status": "passed", "passed": 0, "total": len(checks), "feedback": "All visible tests passed", "failures": []}

try:
    with contextlib.redirect_stdout(io.StringIO()), contextlib.redirect_stderr(io.StringIO()):
        exec(compile(source, "<candidate>", "exec"), namespace)
        if setup:
            exec(compile(setup, "<setup>", "exec"), namespace)
except BaseException as e:
    result["status"] = "syntax_error" if isinstance(e, SyntaxError) else "runtime_error"
    result["feedback"] = f"Failed before tests: {trace(e)}"
else:
    for test in checks:
        try:
            with contextlib.redirect_stdout(io.StringIO()), contextlib.redirect_stderr(io.StringIO()):
                # For simple equality assertions, extract actual/expected values
                # to provide useful execution feedback without reevaluating them.
                tree = ast.parse(test, mode="exec")
                if (len(tree.body) == 1 and isinstance(tree.body[0], ast.Assert) and
                    isinstance(tree.body[0].test, ast.Compare) and
                    len(tree.body[0].test.ops) == 1 and
                    isinstance(tree.body[0].test.ops[0], ast.Eq)):
                    cmp = tree.body[0].test
                    actual = eval(compile(ast.Expression(cmp.left), "<actual>", "eval"), namespace)
                    expected = eval(compile(ast.Expression(cmp.comparators[0]), "<expected>", "eval"), namespace)
                    if actual != expected:
                        raise AssertionError(f"expected {repr(expected)[:220]}, got {repr(actual)[:220]}")
                else:
                    exec(compile(tree, "<test>", "exec"), namespace)
            result["passed"] += 1
        except BaseException as e:
            kind = "wrong_output" if isinstance(e, AssertionError) else "runtime_error"
            result["status"] = kind if result["status"] == "passed" else result["status"]
            result["failures"].append({"test": test[:500], "kind": kind, "message": trace(e)})
            # feedback limited to first failing test to avoid lengthy prompts
            if len(result["failures"]) >= 2:
                break
    if result["failures"]:
        result["feedback"] = "\n".join(
            f"Test: {f['test']}\nError: {f['message']}" for f in result["failures"]
        )[:1600]
print(json.dumps(result, ensure_ascii=False))
'''


def _limits():
    # POSIX-only; Kaggle is Linux. Limits accidental infinite loops and output size.
    resource.setrlimit(resource.RLIMIT_CPU, (3, 4))
    resource.setrlimit(resource.RLIMIT_AS, (1024 * 1024**2, 1024 * 1024**2))
    resource.setrlimit(resource.RLIMIT_FSIZE, (1024 * 1024, 1024 * 1024))


def run_visible_tests(code: str, tests: list[str], setup: str = "", timeout_s: int = 6) -> dict:
    """Execute candidate and visible asserts with timeout.

    Avoid passing hidden benchmark tests here: they must not feed back to the model.
    """
    payload = json.dumps({"code": code, "tests": list(tests), "setup": setup})
    # Pass a small environment, with no authentication tokens inherited.
    env = {"PATH": os.environ.get("PATH", "/usr/bin:/bin"),
           "PYTHONIOENCODING": "utf-8", "PYTHONDONTWRITEBYTECODE": "1"}
    with tempfile.TemporaryDirectory(prefix="gemma_code_") as temp:
        try:
            cp = subprocess.run(
                [sys.executable, "-I", "-S", "-B", "-c", _CHILD],
                input=payload, text=True, stdout=subprocess.PIPE,
                stderr=subprocess.PIPE, cwd=temp,
                env=env, preexec_fn=_limits, timeout=timeout_s,
            )
        except subprocess.TimeoutExpired:
            return {"status": "timeout", "passed": 0, "total": len(tests),
                    "feedback": f"Execution timed out after {timeout_s} seconds", "failures": []}
        if cp.returncode != 0:
            return {"status": "runtime_error", "passed": 0, "total": len(tests),
                    "feedback": f"Runner failed: {cp.stderr[-700:]} (exit={cp.returncode})", "failures": []}
        try:
            output = json.loads(cp.stdout)
            assert output["status"] in {"passed", "wrong_output", "runtime_error", "syntax_error"}
            return output
        except (ValueError, KeyError, AssertionError):
            return {"status": "runner_error", "passed": 0, "total": len(tests),
                    "feedback": f"Malformed runner result: {cp.stdout[-500:]}", "failures": []}


if __name__ == "__main__":
    tests = ["assert find_max([1,5,3]) == 5", "assert find_max([-4, -2]) == -2"]
    wrong = run_visible_tests("def find_max(a):\n    return min(a)\n", tests)
    right = run_visible_tests("def find_max(a):\n    return max(a)\n", tests)
    print("Wrong:", wrong)
    print("Right:", right)
    assert wrong["status"] == "wrong_output"
    assert right["status"] == "passed"

In [ ]:
%%writefile /kaggle/working/agent_loop.py
"""Gemma 4 multi-turn function-repair loop (day-one research prototype)."""
from __future__ import annotations

import re
from typing import Any

from test_runner import run_visible_tests


def extract_python(text: str) -> str:
    """Best effort; keep a record of raw responses for audit."""
    fences = re.findall(r"```(?:python|py)?\s*\n(.*?)```", text, re.S | re.I)
    if fences:
        return fences[-1].strip()
    text = text.strip().replace("<|im_end|>", "").strip()
    # If Gemma adds a preface, discard it before the first code-like line.
    match = re.search(r"(?m)^(?:from\s+\S+\s+import|import\s+\S+|def\s+\w+|class\s+\w+|@\w+)", text)
    return text[match.start():].strip() if match else text


def build_initial_messages(problem: dict) -> list[dict[str, str]]:
    examples = "\n".join(problem.get("test_list", [])[:3])
    return [
        {"role": "system", "content": "Write correct Python code only. Do not include Markdown or explanations. "
            "Include all necessary imports and define the function exactly as requested."},
        {"role": "user", "content": f"Problem:\n{problem['text']}\n\nVisible tests:\n{examples}\n\nWrite Python code that passes these tests."},
    ]


def run_agent(problem: dict, generate_fn, max_attempts: int = 3, max_new_tokens: int = 384) -> dict:
    """generate_fn(messages, max_new_tokens) -> (response_text, input_tokens, output_tokens)."""
    assert 1 <= max_attempts <= 3
    messages = build_initial_messages(problem)
    history = []
    for attempt in range(1, max_attempts + 1):
        raw, prompt_tokens, completion_tokens = generate_fn(messages, max_new_tokens)
        code = extract_python(raw)
        outcome = run_visible_tests(code, problem["test_list"], problem.get("test_setup_code") or "")
        item = {
            "attempt": attempt, "code": code, "raw_output": raw,
            "status": outcome["status"], "feedback": outcome["feedback"],
            "visible_passed": outcome["passed"], "visible_total": outcome["total"],
            "prompt_tokens": prompt_tokens, "completion_tokens": completion_tokens,
        }
        history.append(item)
        if outcome["status"] == "passed":
            break
        messages.append({"role": "assistant", "content": code})
        messages.append({"role": "user", "content": "The previous code FAILED the visible tests.\n"
                        + outcome["feedback"] + "\nProvide a corrected full Python solution. "
                        "Do not repeat the same code. Output code only."})
    return {
        "task_id": int(problem["task_id"]), "problem": problem["text"],
        "solved_visible": history[-1]["status"] == "passed", "history": history,
        "attempts_used": len(history),
        "total_completion_tokens": sum(h["completion_tokens"] for h in history),
        "total_prompt_tokens": sum(h["prompt_tokens"] for h in history),
        "note": "Visible-test success only; not MBPP+ or hidden-test success.",
    }

In [ ]:
%%writefile /kaggle/working/day4_eval.py
"""Day 4: paired, held-out MBPP+ recovery evaluation for Gemma 4.

Generate on visible ORIGINAL MBPP tests. Score with official EvalPlus ONLY
in a separate post-generation subprocess. Never include hidden feedback
in model prompts. Use only on disposable, credential-free runtimes.
"""
from __future__ import annotations

import contextlib
import hashlib
import json
import os
import random
import shutil
import sys
import time
from collections import Counter
from pathlib import Path
from zipfile import ZipFile

from agent_loop import build_initial_messages, extract_python
from test_runner import run_visible_tests

BASE_ID = 'google/gemma-4-E4B-it'
MODES = ('c0', 'c1', 'c2')


def write_json(path, obj):
    p = Path(path); p.parent.mkdir(parents=True, exist_ok=True)
    temp = p.with_suffix(p.suffix + '.tmp')
    temp.write_text(json.dumps(obj, indent=2, ensure_ascii=False), encoding='utf-8')
    temp.replace(p)


def read_jsonl(path):
    p = Path(path)
    if not p.exists():
        return []
    rows = []
    for i, ln in enumerate(p.read_text(encoding='utf-8').splitlines(), 1):
        if ln.strip():
            try:
                rows.append(json.loads(ln))
            except json.JSONDecodeError as exc:
                raise ValueError(f'Corrupt checkpoint {p} at line {i}: {exc}') from exc
    return rows


def append_jsonl(path, row):
    p = Path(path); p.parent.mkdir(parents=True, exist_ok=True)
    with p.open('a', encoding='utf-8') as f:
        f.write(json.dumps(row, ensure_ascii=False) + '\n')
        f.flush()
        os.fsync(f.fileno())


def sha256_bytes(b):
    return hashlib.sha256(b).hexdigest()


def _copy_selected_from_zip(zip_path, base, condition_dirs=False):
    """Safely extract a small allowlist; never do unrestricted extractall."""
    wanted_files = {'mbpp_split_manifest.json', 'training_plan.json'}
    out = Path(base)
    with ZipFile(zip_path, 'r') as z:
        if z.testzip() is not None:
            raise ValueError(f'Corrupt ZIP: {zip_path}')
        for item in z.infolist():
            if item.is_dir():
                continue
            parts = Path(item.filename).parts
            if '__MACOSX' in parts or '..' in parts:
                continue
            name = parts[-1]
            if name in wanted_files:
                (out / name).write_bytes(z.read(item))
            for cond in ('c1', 'c2'):
                if (f'adapter_{cond}' in parts and
                    name in {'adapter_model.safetensors', 'adapter_config.json'}):
                    dst = out / f'adapter_{cond}'
                    dst.mkdir(exist_ok=True)
                    (dst / name).write_bytes(z.read(item))


def restore_inputs(kaggle_root='/kaggle/input', out='/kaggle/working/day4_inputs'):
    """Locate saved Day 2 and Day 3 outputs or attached backup ZIPs.

    Add Day 2 AND Day 3 saved notebook outputs via Kaggle Add Input, or
    attach private Kaggle Datasets containing gemma_day2_backup.zip and
    gemma_day3_backup.zip.
    """
    root = Path(kaggle_root)
    out = Path(out); out.mkdir(parents=True, exist_ok=True)
    required = lambda: all((out / f'adapter_{c}' / 'adapter_model.safetensors').is_file() and
                           (out / f'adapter_{c}' / 'adapter_config.json').is_file()
                           for c in ('c1', 'c2')) and (out/'mbpp_split_manifest.json').is_file()
    if required():
        return out
    if not root.exists():
        raise FileNotFoundError(f'No Kaggle input directory at {root}')
    # Loose files from notebook output: copy adapters and manifest.
    for p in sorted(root.rglob('mbpp_split_manifest.json')):
        if p.is_file():
            shutil.copy2(p, out/'mbpp_split_manifest.json')
            break
    for p in sorted(root.rglob('training_plan.json')):
        if p.is_file():
            shutil.copy2(p, out/'training_plan.json')
            break
    for cond in ('c1', 'c2'):
        configs = sorted(p for p in root.rglob('adapter_config.json') if p.parent.name == f'adapter_{cond}')
        for p in configs:
            w = p.parent/'adapter_model.safetensors'
            if w.is_file():
                dst = out/f'adapter_{cond}'; dst.mkdir(exist_ok=True)
                shutil.copy2(p, dst/p.name)
                shutil.copy2(w, dst/w.name)
                break
    # ZIP backups: only read trusted allowlisted files.
    for pattern in ('gemma_day2_backup*.zip', 'gemma_day3_backup*.zip'):
        for zpath in sorted(root.rglob(pattern)):
            _copy_selected_from_zip(zpath, out)
    if not required():
        missing = [str(p) for p in [out/'mbpp_split_manifest.json',
                  out/'adapter_c1/adapter_config.json',out/'adapter_c1/adapter_model.safetensors',
                  out/'adapter_c2/adapter_config.json',out/'adapter_c2/adapter_model.safetensors'] if not p.exists()]
        raise FileNotFoundError('Attach Day 2 and Day 3 output data. Missing: ' + str(missing))
    a1 = json.loads((out/'adapter_c1/adapter_config.json').read_text())
    a2 = json.loads((out/'adapter_c2/adapter_config.json').read_text())
    assert a1.get('base_model_name_or_path') == BASE_ID and a2.get('base_model_name_or_path') == BASE_ID, 'Adapter base checkpoint mismatch'
    return out


def select_dev_problems(inputs, output, n=30, seed=20261108):
    """Freeze a development set using the original MBPP test split.

    MBPP+ task IDs identify eligible held-out tasks, but NO hidden input/output
    is loaded during generation. Reference code is only used for a local runner
    compatibility check; exclude tasks if that runner cannot execute their
    reference code correctly. Report these exclusions.
    """
    from datasets import load_dataset
    from tqdm.auto import tqdm
    inputs = Path(inputs); output = Path(output)
    output.mkdir(parents=True, exist_ok=True)
    manifest = json.loads((inputs/'mbpp_split_manifest.json').read_text())
    dev_path = output/'eval_manifest.json'
    ds = load_dataset('google-research-datasets/mbpp', 'full')
    rows = {int(row['task_id']): dict(row) for row in ds['test']}
    eval_ids = {int(x) for x in manifest['evalplus_mbppplus_ids']}
    trained_ids = set(manifest['mbppplus_decontaminated']['train']['task_ids'])
    candidate_ids = sorted(set(rows).intersection(eval_ids) - trained_ids)
    assert len(candidate_ids) >= n, f'Only {len(candidate_ids)} compatible ID candidates, requested {n}'
    if dev_path.exists():
        frozen = json.loads(dev_path.read_text())
        assert frozen['requested_n'] == n and frozen['seed'] == seed, 'Dev manifest already frozen for different settings; use new output directory'
        chosen = frozen['task_ids']
        assert set(chosen).issubset(candidate_ids), 'Frozen task set no longer matches dataset'
        return [rows[tid] for tid in chosen], frozen
    order = list(candidate_ids)
    random.Random(seed).shuffle(order)
    chosen, excluded = [], []
    for tid in tqdm(order, desc='Check visible-test runner compatibility', unit='task'):
        row = rows[tid]
        oracle = run_visible_tests(row['code'], row['test_list'], row.get('test_setup_code') or '')
        if oracle['status'] == 'passed':
            chosen.append(tid)
            if len(chosen) == n:
                break
        else:
            excluded.append({'task_id': tid, 'reason': oracle['status']})
    if len(chosen) < n:
        raise ValueError(f'Only {len(chosen)} local-runner-compatible tasks. Lower requested n or inspect runner.')
    frozen = {
        'name':'MBPP+ overlap with original MBPP test split, held-out dev set',
        'task_ids': chosen, 'seed': seed, 'requested_n': n,
        'candidate_count': len(candidate_ids),
        'runner_exclusions': excluded,
        'training_overlap': sorted(set(chosen).intersection(trained_ids)),
        'visible_tests': 'original MBPP test_list',
        'hidden_tests': 'official EvalPlus base_input/plus_input, scored only after generation',
        'do_not_use_for_final_claims': 'Use separate untouched test tasks in final evaluation after development is frozen.',
    }
    assert not frozen['training_overlap']
    write_json(dev_path, frozen)
    return [rows[tid] for tid in chosen], frozen


def load_all_adapters(inputs, model_id=BASE_ID):
    import torch
    from transformers import AutoModelForMultimodalLM, AutoProcessor, BitsAndBytesConfig
    from peft import PeftModel
    quant = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type='nf4',
        bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=torch.float16)
    print('Loading Gemma 4 in 4-bit on GPU 0 ...', flush=True)
    base = AutoModelForMultimodalLM.from_pretrained(
        model_id, quantization_config=quant, device_map={'':0}, dtype=torch.float16)
    base.config.use_cache = True
    processor = AutoProcessor.from_pretrained(model_id)
    model = PeftModel.from_pretrained(base, str(Path(inputs)/'adapter_c1'),
        adapter_name='c1', is_trainable=False)
    model.load_adapter(str(Path(inputs)/'adapter_c2'), adapter_name='c2', is_trainable=False)
    model.eval()
    model.set_adapter('c1')
    assert {'c1','c2'} <= set(model.peft_config.keys()), f'Adapters not loaded: {model.peft_config.keys()}'
    print('Loaded C1 + C2. GPU0 allocated:', round(torch.cuda.memory_allocated(0)/1024**3, 2), 'GiB', flush=True)
    return model, processor


def generate_code(model, processor, messages, condition, max_new_tokens=384):
    """Deterministic inference; C0 disables all adapters temporarily."""
    import torch
    if condition not in MODES:
        raise ValueError(condition)
    if condition in ('c1','c2'):
        model.set_adapter(condition)
        ctx = contextlib.nullcontext()
    else:
        ctx = model.disable_adapter()
    inputs = processor.apply_chat_template(
        messages, tokenize=True, return_dict=True, return_tensors='pt',
        add_generation_prompt=True, enable_thinking=False).to('cuda:0')
    n = int(inputs['input_ids'].shape[1])
    started = time.perf_counter()
    with ctx, torch.inference_mode():
        out = model.generate(**inputs, do_sample=False, max_new_tokens=max_new_tokens,
            pad_token_id=processor.tokenizer.pad_token_id)
    new = out[0][n:]
    return {
        'raw': processor.decode(new, skip_special_tokens=True),
        'prompt_tokens':n, 'completion_tokens':int(len(new)),
        'generation_seconds':round(time.perf_counter()-started, 3),
    }


def record_base_initials(problems, model, processor, output, max_new_tokens=384):
    from tqdm.auto import tqdm
    output = Path(output); output.mkdir(parents=True, exist_ok=True)
    log = output/'base_initials.jsonl'
    existing = {int(r['task_id']):r for r in read_jsonl(log)}
    if len(existing)!=len(read_jsonl(log)):
        raise ValueError('Duplicate base initial task IDs found in checkpoint')
    pending = [p for p in problems if int(p['task_id']) not in existing]
    with tqdm(total=len(problems), initial=len(problems)-len(pending),
              desc='C0 first attempts', unit='problem', dynamic_ncols=True) as bar:
        for p in pending:
            tid = int(p['task_id'])
            bar.set_postfix(task=tid)
            print(f'Generating initial code for MBPP {tid}', flush=True)
            g = generate_code(model, processor, build_initial_messages(p), 'c0', max_new_tokens)
            code = extract_python(g['raw'])
            check = run_visible_tests(code, p['test_list'], p.get('test_setup_code') or '')
            rec = {'task_id':tid,'condition':'c0','attempt':1,'code':code,
                   'visible_status':check['status'], 'visible_feedback':check['feedback'],
                   'visible_passed':check.get('passed',0), 'visible_total':check.get('total',0),
                   **g}
            append_jsonl(log, rec)
            existing[tid]=rec
            bar.update()
            bar.set_postfix(task=tid, visible=check['status'], sec=g['generation_seconds'])
    return existing


def _feedback_message(feedback):
    return 'The previous code FAILED the visible tests.\n' + feedback + '\nProvide a corrected full Python solution. Do not repeat the same code. Output code only.'


def record_fixed_corrections(problems, initials, model, processor, output, max_new_tokens=384, max_corrections=1):
    """Same failed C0 code and same initial feedback for C0/C1/C2.

    No corrections are issued after the visible tests pass, regardless of
    eventual hidden-test success/failure. Hidden results are never in prompts.
    """
    from tqdm.auto import tqdm
    output = Path(output); output.mkdir(parents=True, exist_ok=True)
    log=output/'fixed_corrections.jsonl'
    existing_rows=read_jsonl(log)
    existing={(int(r['task_id']),r['condition']):r for r in existing_rows}
    if len(existing)!=len(existing_rows):
        raise ValueError('Duplicate fixed-state records in checkpoint')
    tasks=[p for p in problems if initials[int(p['task_id'])]['visible_status']!='passed']
    count=len(tasks)*len(MODES)
    assert tasks, 'No visible failures in chosen tasks. Increase number of evaluation tasks; do not change their IDs after looking at hidden results.'
    pending=sum((int(p['task_id']),c) not in existing for p in tasks for c in MODES)
    with tqdm(total=count,initial=count-pending,desc='Paired recovery conditions',unit='condition',dynamic_ncols=True) as bar:
        for p in tasks:
            tid=int(p['task_id'])
            init=initials[tid]
            if init['visible_status']=='runner_error':
                raise ValueError(f'Runner error on task {tid}; inspect before evaluation')
            for condition in MODES:
                key=(tid,condition)
                if key in existing:
                    continue
                messages=build_initial_messages(p)
                messages.append({'role':'assistant','content':init['code']})
                messages.append({'role':'user','content':_feedback_message(init['visible_feedback'])})
                attempts=[]
                for retry in range(1,max_corrections+1):
                    print(f'Fixing MBPP {tid}: {condition}, correction {retry}/{max_corrections}',flush=True)
                    g=generate_code(model,processor,messages,condition,max_new_tokens)
                    code=extract_python(g['raw'])
                    chk=run_visible_tests(code,p['test_list'],p.get('test_setup_code') or '')
                    attempts.append({'correction_index':retry,'code':code,
                                     'visible_status':chk['status'], 'visible_feedback':chk['feedback'],
                                     'visible_passed':chk.get('passed',0),
                                     'visible_total':chk.get('total',0), **g})
                    if chk['status']=='passed':
                        break
                    if retry < max_corrections:
                        messages.extend([
                            {'role':'assistant','content':code},
                            {'role':'user','content':_feedback_message(chk['feedback'])}])
                rec={'task_id':tid,'condition':condition,
                     'starting_code':init['code'],
                     'starting_feedback':init['visible_feedback'],
                     'starting_visible_status':init['visible_status'],
                     'attempts':attempts,
                     'recovered_visible':any(a['visible_status']=='passed' for a in attempts),
                     'num_corrections':len(attempts),
                     'total_completion_tokens':sum(a['completion_tokens'] for a in attempts),
                     'total_prompt_tokens':sum(a['prompt_tokens'] for a in attempts)}
                append_jsonl(log,rec);existing[key]=rec
                bar.update()
                bar.set_postfix(task=tid,condition=condition,
                                visible=rec['recovered_visible'])
    return existing


def prepare_offline_scoring(problems, output):
    """Create evaluation inputs after all completions were generated."""
    output=Path(output)
    ids={int(p['task_id']) for p in problems}
    initials=read_jsonl(output/'base_initials.jsonl')
    fixed=read_jsonl(output/'fixed_corrections.jsonl')
    assert {int(r['task_id']) for r in initials} == ids, 'Initial generation incomplete'
    failed={int(r['task_id']) for r in initials if r['visible_status']!='passed'}
    for cond in MODES:
        assert {int(r['task_id']) for r in fixed if r['condition']==cond} == failed, f'Missing fixed corrections for {cond}'
    rows=[]
    for r in initials:
        rows.append({'score_key':f'initial:{int(r["task_id"])}',
                     'task_id':int(r['task_id']), 'condition':'c0',
                     'stage':'initial', 'attempt':1,'code':r['code']})
    for r in fixed:
        for a in r['attempts']:
            rows.append({'score_key':f'fixed:{r["condition"]}:{int(r["task_id"])}:{a["correction_index"]}',
                         'task_id':int(r['task_id']), 'condition':r['condition'],
                         'stage':'fixed_recovery','attempt':a['correction_index'],
                         'code':a['code']})
    assert len(rows)==len({r['score_key'] for r in rows}), 'Duplicate score keys'
    p=output/'scoring_inputs.jsonl'
    tmp=p.with_suffix('.jsonl.tmp')
    tmp.write_text(''.join(json.dumps(r,ensure_ascii=False)+'\n' for r in rows),encoding='utf-8')
    tmp.replace(p)
    return p


def _rate(arr):
    return sum(arr)/len(arr) if arr else float('nan')


def paired_bootstrap(c1,c2,n_boot=3000,seed=42):
    assert len(c1)==len(c2)>0
    diffs=[int(b)-int(a) for a,b in zip(c1,c2)]
    rng=random.Random(seed)
    draws=[]
    for _ in range(n_boot):
        draws.append(sum(diffs[rng.randrange(len(diffs))] for _ in diffs)/len(diffs))
    draws.sort()
    return {'difference_c2_minus_c1':sum(diffs)/len(diffs),
            'ci_low':draws[int(0.025*n_boot)],
            'ci_high':draws[min(n_boot-1,int(0.975*n_boot))],
            'n_pairs':len(diffs),
            'discordant_c2_only':sum(d==1 for d in diffs),
            'discordant_c1_only':sum(d==-1 for d in diffs)}


def analyze_official(problems, output):
    output=Path(output)
    initials=read_jsonl(output/'base_initials.jsonl')
    corrections=read_jsonl(output/'fixed_corrections.jsonl')
    official=read_jsonl(output/'official_scores.jsonl')
    im={r['score_key']:r for r in official}
    expected_keys={r['score_key'] for r in read_jsonl(output/'scoring_inputs.jsonl')}
    if set(im)!=expected_keys or len(im)!=len(official):
        raise ValueError('Incomplete/duplicate official score log. Rerun scorer.')
    failed_ids=sorted(int(r['task_id']) for r in initials if r['visible_status']!='passed')
    f={(int(r['task_id']),r['condition']):r for r in corrections}
    successes={}
    for cond in MODES:
        hits=[];visible=[];tokens=[]
        for tid in failed_ids:
            r=f[(tid,cond)]
            hits.append(any(im[f'fixed:{cond}:{tid}:{a["correction_index"]}']['mbppplus_pass'] for a in r['attempts']))
            visible.append(r['recovered_visible'])
            tokens.append(r['total_completion_tokens'])
        successes[cond]={'official_recovery':sum(hits), 'n_failed_states':len(failed_ids),
             'recovery_rate':_rate(hits), 'visible_recovery_rate':_rate(visible),
             'mean_completion_tokens':_rate(tokens)}
    c1=[];c2=[]
    for tid in failed_ids:
        for arr,cond in ((c1,'c1'),(c2,'c2')):
            r=f[(tid,cond)]
            arr.append(any(im[f'fixed:{cond}:{tid}:{a["correction_index"]}']['mbppplus_pass'] for a in r['attempts']))
    base_pass=sum(im[f'initial:{int(r["task_id"])}']['mbppplus_pass'] for r in initials)
    summary={
        'benchmark': 'EvalPlus MBPP+ official base + plus oracle',
        'dev_only':True, 'n_dev_tasks':len(problems),
        'n_fixed_failed_states':len(failed_ids),
        'base_first_attempt_official_pass':base_pass,
        'base_first_attempt_official_rate':base_pass/len(initials),
        'fixed_state_results':successes,
        'paired_C2_vs_C1':paired_bootstrap(c1,c2),
        'methodology_note':'Fixed starting states are first-attempt visible failures from C0; hidden tests are only scored after all generations and never returned as feedback.',
    }
    write_json(output/'day4_summary.json',summary)
    return summary

In [ ]:
%%writefile /kaggle/working/day4_score.py
"""Offline official EvalPlus MBPP+ scoring for Day 4.

Run ONLY after generation, in a separate CPU subprocess. No hidden feedback
is supplied to the coding agent. IMPORTANT: EvalPlus executes generated code;
run only in an isolated, credential-free container/runtime.
"""
from __future__ import annotations
import argparse, hashlib, json, sys
from pathlib import Path
from tqdm.auto import tqdm

from day4_eval import append_jsonl, read_jsonl, sha256_bytes


def score(input_jsonl, output_jsonl):
    # Import lazily: model-generation process does not load hidden oracle.
    from evalplus.data import get_mbpp_plus, get_mbpp_plus_hash
    from evalplus.evaluate import check_correctness, get_groundtruth
    from evalplus.eval._special_oracle import MBPP_OUTPUT_NOT_NONE_TASKS
    source=Path(input_jsonl)
    dest=Path(output_jsonl)
    requests=read_jsonl(source)
    if not requests:
        raise RuntimeError(f'No scoring requests: {source}')
    score_keys=[r['score_key'] for r in requests]
    if len(set(score_keys))!=len(score_keys):
        raise ValueError('Duplicate score keys')
    cache_inputs=dest.with_suffix('.source_sha256')
    hash_input=sha256_bytes(source.read_bytes())
    if cache_inputs.exists() and cache_inputs.read_text().strip()!=hash_input:
        raise RuntimeError('Score requests changed since scoring began; do not mix checkpoints')
    cache_inputs.parent.mkdir(parents=True,exist_ok=True)
    cache_inputs.write_text(hash_input+'\n')
    old=read_jsonl(dest)
    if len(old)!=len({r['score_key'] for r in old}):
        raise ValueError('Duplicate score checkpoints')
    done={r['score_key']:r for r in old}
    if not set(done).issubset(score_keys):
        raise ValueError('Checkpoint contains scores for different task set')
    problems_all=get_mbpp_plus()
    key_by_id = {int(str(key).split('/')[-1]): key for key in problems_all}
    requested_ids = {int(r['task_id']) for r in requests}
    assert requested_ids.issubset(key_by_id), 'Some selected IDs are absent from official EvalPlus MBPP+'
    ids = {key_by_id[tid] for tid in requested_ids}
    problems={tid:problems_all[tid] for tid in ids}
    print('Loaded',len(problems),'official EvalPlus MBPP+ problems',flush=True)
    # Hash includes dataset version AND selected subset: NEVER corrupt full-benchmark cache.
    subhash=hashlib.sha256((get_mbpp_plus_hash() + '|' + ','.join(sorted(ids))).encode()).hexdigest()[:24]
    expected=get_groundtruth(problems,'day4_dev_subset_'+subhash,MBPP_OUTPUT_NOT_NONE_TASKS)
    remaining=[r for r in requests if r['score_key'] not in done]
    with tqdm(total=len(requests),initial=len(done),desc='Official EvalPlus scoring',unit='solution',dynamic_ncols=True) as bar:
        for r in remaining:
            tid=key_by_id[int(r['task_id'])]
            try:
                result=check_correctness(
                    'mbpp', 0, problems[tid], r['code'], expected[tid],
                    base_only=False, fast_check=True,
                )
                base_status=result['base'][0]
                plus_status=result['plus'][0]
            except Exception as exc:
                # Treat scorer exceptions as infrastructure issues, not incorrect code.
                raise RuntimeError(f'Official evaluator failed on {r["score_key"]}: {exc}') from exc
            rec={'score_key':r['score_key'],'task_id':int(r['task_id']),
                 'stage':r['stage'],'condition':r['condition'],
                 'evalplus_base_status':base_status,
                 'evalplus_plus_status':plus_status,
                 'mbppplus_pass':base_status=='pass' and plus_status=='pass'}
            append_jsonl(dest,rec)
            bar.update()
            bar.set_postfix(task=tid,pass_plus=rec['mbppplus_pass'])
    print('Finished:',len(read_jsonl(dest)),'official scores saved at',dest,flush=True)


if __name__=='__main__':
    parser=argparse.ArgumentParser()
    parser.add_argument('--requests',required=True)
    parser.add_argument('--output',required=True)
    args=parser.parse_args()
    score(args.requests,args.output)

In [ ]:
%%writefile /kaggle/working/day5_eval.py
"""Day 5: extended *development-only* repair analysis.

Generation receives original MBPP visible tests only. EvalPlus hidden tests are
NEVER imported by this module; score in a separate CPU-only process afterward.
Run candidate code only in an isolated credential-free environment.
"""
from __future__ import annotations

import ast
import difflib
import hashlib
import json
import random
import shutil
from collections import Counter, defaultdict
from pathlib import Path
from zipfile import ZipFile

from tqdm.auto import tqdm

from day4_eval import (
    BASE_ID, MODES, append_jsonl, read_jsonl, write_json, generate_code,
    record_base_initials, record_fixed_corrections, prepare_offline_scoring,
    paired_bootstrap, _feedback_message,
)
from agent_loop import build_initial_messages, extract_python
from test_runner import run_visible_tests


def _safe_zip_extract(zip_path, dest, prefix, allow_names):
    dest = Path(dest)
    with ZipFile(zip_path) as z:
        if z.testzip() is not None:
            raise ValueError(f'ZIP failed integrity check: {zip_path}')
        for member in z.infolist():
            parts = Path(member.filename).parts
            if member.is_dir() or '..' in parts or '__MACOSX' in parts:
                continue
            if len(parts) == 2 and parts[0] == prefix and parts[-1] in allow_names:
                dest.mkdir(parents=True, exist_ok=True)
                (dest / parts[-1]).write_bytes(z.read(member))


def restore_day4_results(kaggle_root='/kaggle/input', out='/kaggle/working/day5_prior_day4'):
    """Restore Day 4 development results from saved notebook outputs or ZIP."""
    root, out = Path(kaggle_root), Path(out)
    out.mkdir(parents=True, exist_ok=True)
    names = {
        'eval_manifest.json', 'base_initials.jsonl', 'fixed_corrections.jsonl',
        'official_scores.jsonl', 'scoring_inputs.jsonl', 'day4_summary.json',
        'evaluation_protocol.json',
    }
    if not (out / 'day4_summary.json').exists():
        for candidate in sorted(root.rglob('gemma_day4_backup*.zip')):
            _safe_zip_extract(candidate, out, 'day4_outputs', names)
        # Loose output files from an attached saved notebook.
        for filename in names:
            if (out / filename).exists():
                continue
            for p in sorted(root.rglob(filename)):
                if 'day4_outputs' in p.parts and p.is_file():
                    shutil.copy2(p, out / filename)
                    break
    missing = sorted(n for n in names if not (out / n).is_file())
    if missing:
        raise FileNotFoundError(
            'Attach Day 4 saved outputs or gemma_day4_backup.zip via Add Input. Missing: ' + str(missing))
    manifest = json.loads((out/'eval_manifest.json').read_text())
    old_initials = read_jsonl(out/'base_initials.jsonl')
    old_fixed = read_jsonl(out/'fixed_corrections.jsonl')
    old_scores = read_jsonl(out/'official_scores.jsonl')
    old_requests = read_jsonl(out/'scoring_inputs.jsonl')
    ids = {int(x) for x in manifest['task_ids']}
    assert len(ids) == len(old_initials) == len(manifest['task_ids'])
    assert {int(r['task_id']) for r in old_initials} == ids
    failed = {int(r['task_id']) for r in old_initials if r['visible_status'] != 'passed'}
    assert {(int(r['task_id']),r['condition']) for r in old_fixed} == {(i,c) for i in failed for c in MODES}
    assert len({r['score_key'] for r in old_scores}) == len(old_scores)
    assert {r['score_key'] for r in old_scores} == {r['score_key'] for r in old_requests}
    print(f'Restored Day 4: {len(ids)} tasks; {len(failed)} initial visible failures; {len(old_scores)} scores')
    return out, manifest



def restore_day5_checkpoint(kaggle_root='/kaggle/input', output='/kaggle/working/day5_outputs'):
    """Optionally restore saved Day5 partial/full backup from attached Kaggle Input.

    Never overwrite an existing local result with different bytes. The caller
    still checks frozen configuration and all task IDs before running anything.
    """
    root,out=Path(kaggle_root),Path(output)
    out.mkdir(parents=True,exist_ok=True)
    restored=0
    for pattern in ('day5_generation_checkpoint*.zip','gemma_day5_backup*.zip'):
        for archive in sorted(root.rglob(pattern)):
            with ZipFile(archive) as z:
                if z.testzip() is not None: raise ValueError(f'Corrupt resume ZIP: {archive}')
                for item in z.infolist():
                    parts=Path(item.filename).parts
                    if item.is_dir() or '..' in parts or len(parts)<2 or parts[0]!='day5_outputs':
                        continue
                    name=Path(*parts[1:])
                    if name.is_absolute():raise ValueError(f'Unsafe ZIP member: {name}')
                    target=out/name
                    content=z.read(item)
                    if target.exists():
                        if target.read_bytes()!=content:
                            raise RuntimeError(f'Checkpoint conflict: {target}. Reconcile, do not overwrite.')
                        continue
                    target.parent.mkdir(parents=True,exist_ok=True)
                    target.write_bytes(content)
                    restored+=1
    if restored:print(f'Restored {restored} Day 5 checkpoint files from Kaggle input.')
    return restored

def freeze_partitions(inputs, old_dir, output, n_new=80, n_final=70, seed=20261109):
    """Choose dev extension BEFORE inference; reserve untouched final IDs now.

    Use only original MBPP prompts/test_list and cached MBPP+ task IDs from Day 2.
    A reference-runner compatibility check is done on *dev* candidates only.
    Final task IDs are merely reserved, never loaded into generation/scoring.
    """
    from datasets import load_dataset
    output = Path(output); output.mkdir(parents=True, exist_ok=True)
    frozen_path = output/'day5_partition_manifest.json'
    m = json.loads((Path(inputs)/'mbpp_split_manifest.json').read_text())
    old_m = json.loads((Path(old_dir)/'eval_manifest.json').read_text())
    old_ids = {int(tid) for tid in old_m['task_ids']}
    train_ids = set(map(int, m['mbppplus_decontaminated']['train']['task_ids']))
    mbpp_test = load_dataset('google-research-datasets/mbpp', 'full')['test']
    rows = {int(r['task_id']):dict(r) for r in mbpp_test}
    eligible = sorted((set(rows) & set(map(int,m['evalplus_mbppplus_ids']))) - train_ids - old_ids)
    assert len(eligible) >= n_new+n_final, f'Only {len(eligible)} unseen eligible tasks for {n_new}+{n_final}'
    if frozen_path.is_file():
        plan = json.loads(frozen_path.read_text())
        assert plan['seed']==seed and plan['n_new_requested']==n_new and plan['n_final_reserved']==n_final
        dev, final = set(plan['new_dev_ids']), set(plan['final_holdout_ids'])
        assert len(dev)==n_new and len(final)==n_final and not(dev & final) and not(dev & old_ids)
        assert (dev|final) <= set(eligible)
        return [rows[i] for i in plan['new_dev_ids']],plan
    rng = random.Random(seed)
    rng.shuffle(eligible)
    chosen, excluded = [], []
    # Deterministic oracle-compatibility filter for dev only; we never inspect candidate model outputs here.
    for tid in tqdm(eligible, desc='Freeze compatible dev task list', unit='task'):
        oracle = run_visible_tests(rows[tid]['code'],rows[tid]['test_list'],rows[tid].get('test_setup_code') or '')
        if oracle['status']=='passed':
            chosen.append(tid)
        else:
            excluded.append({'task_id':tid,'reference_status':oracle['status']})
        if len(chosen)==n_new:
            break
    if len(chosen)<n_new: raise RuntimeError('Not enough runner-compatible development tasks.')
    final = [tid for tid in eligible if tid not in set(chosen)][:n_final]
    if len(final)!=n_final: raise RuntimeError('Not enough held-out tasks to reserve.')
    plan = {
        'name':'Day 5 extended-development partition, FINAL set never used in this notebook',
        'seed':seed, 'n_new_requested':n_new, 'n_final_reserved':n_final,
        'prior_day4_dev_ids':sorted(old_ids), 'new_dev_ids':chosen,
        'final_holdout_ids':final,'n_remaining_eligible':len(eligible)-len(chosen)-len(final),
        'dev_runner_exclusions':excluded,
        'training_overlap':sorted(train_ids & (old_ids | set(chosen) | set(final))),
        'protocol':'C0/C1/C2 on new dev IDs, reuse Day 4 dev; no final-holdout evaluation or training.',
    }
    assert not plan['training_overlap']
    write_json(frozen_path,plan)
    return [rows[i] for i in chosen],plan


def freeze_e2e_ids(new_problems, output, n=30, seed=20261110):
    output=Path(output); p=output/'end_to_end_manifest.json'
    ids=[int(x['task_id']) for x in new_problems]
    if p.exists():
        obj=json.loads(p.read_text())
        assert obj['seed']==seed and obj['requested_n']==n and set(obj['task_ids'])<=set(ids)
        return obj['task_ids']
    if n>len(ids): raise ValueError('E2E_N exceeds new development problems')
    rng=random.Random(seed); rng.shuffle(ids)
    chosen=ids[:n]
    write_json(p,{'seed':seed,'requested_n':n,'task_ids':chosen,
                'sampling':'Unconditional random dev subset; chosen before generation, not on observed failure'})
    return chosen


def record_e2e(problems, model, processor, output, max_new_tokens=384, max_attempts=3):
    """Each condition independently starts from problem only, uses feedback after failures."""
    output=Path(output); output.mkdir(parents=True,exist_ok=True)
    log=output/'end_to_end.jsonl'
    previous=read_jsonl(log)
    existing={(int(r['task_id']),r['condition']):r for r in previous}
    assert len(existing)==len(previous), 'Duplicate E2E checkpoint entries'
    total=len(problems)*len(MODES)
    with tqdm(total=total,initial=len(existing),desc='End-to-end 1–3 attempts',unit='condition',dynamic_ncols=True) as bar:
        for p in problems:
            tid=int(p['task_id'])
            for condition in MODES:
                key=(tid,condition)
                if key in existing: continue
                msgs=build_initial_messages(p)
                attempts=[]
                for i in range(1,max_attempts+1):
                    bar.set_postfix(task=tid,condition=condition,attempt=i)
                    g=generate_code(model,processor,msgs,condition,max_new_tokens)
                    code=extract_python(g['raw'])
                    chk=run_visible_tests(code,p['test_list'],p.get('test_setup_code') or '')
                    if chk['status']=='runner_error':
                        raise RuntimeError(f'Visible runner infrastructure error on {tid}/{condition}/{i}')
                    attempts.append({'attempt':i,'code':code,'visible_status':chk['status'],
                                     'visible_feedback':chk['feedback'], **g})
                    if chk['status']=='passed':break
                    if i<max_attempts:
                        msgs.extend([{'role':'assistant','content':code},
                                     {'role':'user','content':_feedback_message(chk['feedback'])}])
                rec={'task_id':tid,'condition':condition,'attempts':attempts,
                     'total_completion_tokens':sum(int(a['completion_tokens']) for a in attempts),
                     'total_prompt_tokens':sum(int(a['prompt_tokens']) for a in attempts),
                     'num_executions':len(attempts),'final_visible_pass':attempts[-1]['visible_status']=='passed'}
                append_jsonl(log,rec);existing[key]=rec;bar.update()
    return existing


def record_no_feedback(problems, initials, model, processor, output, max_new_tokens=384):
    """C2 from same failed code but vague 'failed' message; no concrete test outputs."""
    output=Path(output);log=output/'no_feedback_c2.jsonl'
    rows=read_jsonl(log);existing={int(r['task_id']):r for r in rows}
    assert len(rows)==len(existing)
    failures=[p for p in problems if initials[int(p['task_id'])]['visible_status']!='passed']
    with tqdm(total=len(failures),initial=len(existing),desc='C2 no-feedback ablation',unit='failure') as bar:
        for p in failures:
            tid=int(p['task_id'])
            if tid in existing: continue
            start=initials[tid]
            msgs=build_initial_messages(p)+[
                {'role':'assistant','content':start['code']},
                {'role':'user','content':'The previous code failed a visible test. Try a corrected full Python solution. Output code only.'},
            ]
            g=generate_code(model,processor,msgs,'c2',max_new_tokens)
            code=extract_python(g['raw'])
            chk=run_visible_tests(code,p['test_list'],p.get('test_setup_code') or '')
            if chk['status']=='runner_error':raise RuntimeError('Runner error on no-feedback ablation')
            r={'task_id':tid,'condition':'c2_no_feedback','code':code,'visible_status':chk['status'],
               'visible_feedback':chk['feedback'],**g}
            append_jsonl(log,r);existing[tid]=r;bar.update()
    return existing


def freeze_humaneval(output, n=20, seed=20261111):
    """Load ORIGINAL public HumanEval prompts; do not load HumanEval+ hidden tests."""
    from evalplus.data.humaneval import get_human_eval
    out=Path(output);p=out/'humaneval_manifest.json'
    tasks=get_human_eval()
    ids=sorted(tasks)
    if p.exists():
        m=json.loads(p.read_text())
        assert m['n_tasks']==n and m['seed']==seed and set(m['task_ids'])<=set(ids)
        return {tid:tasks[tid] for tid in m['task_ids']}
    if n>len(ids):raise ValueError('n exceeds HumanEval task count')
    rng=random.Random(seed);rng.shuffle(ids)
    selected=ids[:n]
    write_json(p,{'n_tasks':n,'seed':seed,'task_ids':selected,
                  'source':'Original public HumanEval prompt (NOT HumanEval+ tests)',
                  'evaluation':'Single-shot, separate offline official HumanEval+ scoring. Exploratory transfer.'})
    return {tid:tasks[tid] for tid in selected}


def record_humaneval(problems, model, processor, output, max_new_tokens=384):
    from day4_eval import generate_code
    output=Path(output);log=output/'humaneval_first_attempts.jsonl'
    rows=read_jsonl(log);existing={(r['task_id'],r['condition']):r for r in rows}
    assert len(existing)==len(rows)
    with tqdm(total=len(problems)*len(MODES),initial=len(existing),
              desc='HumanEval transfer (first attempt only)',unit='condition') as bar:
        for tid,p in problems.items():
            for condition in MODES:
                key=(tid,condition)
                if key in existing:continue
                prompt='Implement the following Python function. Return a complete, runnable Python implementation including its function definition, not just the body. Output code only.\n\n'+p['prompt']
                msgs=[{'role':'user','content':prompt}]
                g=generate_code(model,processor,msgs,condition,max_new_tokens)
                code=extract_python(g['raw'])
                rec={'task_id':tid,'condition':condition,'code':code,**g}
                append_jsonl(log,rec);existing[key]=rec;bar.update();bar.set_postfix(task=tid,condition=condition)
    return existing


def create_scoring_requests(new_problems, output, include_e2e=True, include_ablation=True,
                            include_humaneval=True):
    """Freeze ALL requests once before invoking any hidden test scoring."""
    output=Path(output)
    # day4_eval.prepare_offline_scoring checks fixed pair completeness first.
    new_path=prepare_offline_scoring(new_problems,output)
    mbpp=read_jsonl(new_path)
    task_ids={int(p['task_id']) for p in new_problems}
    if include_e2e:
        e2e=read_jsonl(output/'end_to_end.jsonl')
        expected={(tid,c) for tid in json.loads((output/'end_to_end_manifest.json').read_text())['task_ids'] for c in MODES}
        assert {(int(r['task_id']),r['condition']) for r in e2e} == expected, 'E2E incomplete'
        for r in e2e:
            for a in r['attempts']:
                mbpp.append({'score_key':f'e2e:{r["condition"]}:{r["task_id"]}:{a["attempt"]}',
                             'task_id':int(r['task_id']),'condition':r['condition'],
                             'stage':'end_to_end','attempt':a['attempt'],'code':a['code']})
    if include_ablation:
        ab=read_jsonl(output/'no_feedback_c2.jsonl')
        initials=read_jsonl(output/'base_initials.jsonl')
        failed={int(x['task_id']) for x in initials if x['visible_status']!='passed'}
        assert {int(x['task_id']) for x in ab}==failed, 'Ablation incomplete'
        for r in ab:
            mbpp.append({'score_key':f'no_feedback:c2:{r["task_id"]}',
                         'task_id':int(r['task_id']),'condition':'c2_no_feedback',
                         'stage':'no_feedback','attempt':1,'code':r['code']})
    assert len(mbpp)==len({r['score_key'] for r in mbpp}), 'Duplicate MBPP scorer keys'
    path=output/'scoring_inputs.jsonl'
    path.write_text(''.join(json.dumps(r,ensure_ascii=False)+'\n' for r in mbpp))
    human=[]
    if include_humaneval:
        h=read_jsonl(output/'humaneval_first_attempts.jsonl')
        m=json.loads((output/'humaneval_manifest.json').read_text())
        assert {(r['task_id'],r['condition']) for r in h} == {(t,c) for t in m['task_ids'] for c in MODES}
        for r in h:
            human.append({'score_key':f'humaneval:{r["condition"]}:{r["task_id"]}',
                          'task_id':r['task_id'],'condition':r['condition'],
                          'stage':'humaneval','code':r['code']})
    human_path=output/'humaneval_scoring_inputs.jsonl'
    human_path.write_text(''.join(json.dumps(r,ensure_ascii=False)+'\n' for r in human))
    write_json(output/'scoring_protocol.json', {
        'mbpp_requests':len(mbpp),'humaneval_requests':len(human),
        'generation_frozen_before_official_scoring':True,
        'source':'Original MBPP visible tests; original HumanEval public prompts',
        'hidden_use':'EvalPlus base+plus in separate CPU process only',
        'do_not_train_on_this_split':True,
    })
    return path,human_path


def _pass(scores,key):
    if key not in scores:
        raise KeyError(f'Missing official score: {key}')
    return bool(scores[key]['mbppplus_pass'])


def _normalize(code):
    try:
        return ast.unparse(ast.parse(code)).strip()
    except (SyntaxError,ValueError):
        return '\n'.join(line.strip() for line in str(code).splitlines() if line.strip())


def _repeat_stats(attempt_codes, statuses, threshold=0.95):
    exact=near=0
    for i in range(1,len(attempt_codes)):
        if statuses[i]=='passed':continue
        a,b=_normalize(attempt_codes[i-1]),_normalize(attempt_codes[i])
        if a==b:
            exact+=1
        elif difflib.SequenceMatcher(None,a,b).ratio()>=threshold:
            near+=1
    return exact,near


def _boot(c1,c2,seed=42):
    if not c1:return {'n_pairs':0,'difference_c2_minus_c1':None,'ci_low':None,'ci_high':None}
    return paired_bootstrap(c1,c2,seed=seed)


def analyze(old_dir,new_dir, include_e2e=True,include_ablation=True,include_humaneval=True):
    """Compute fixed-state, failure, budget, and compute statistics from OFFICIAL scores."""
    from statistics import mean
    import csv
    old_dir,new_dir=Path(old_dir),Path(new_dir)
    old_init=read_jsonl(old_dir/'base_initials.jsonl')
    new_init=read_jsonl(new_dir/'base_initials.jsonl')
    old_fix=read_jsonl(old_dir/'fixed_corrections.jsonl')
    new_fix=read_jsonl(new_dir/'fixed_corrections.jsonl')
    old_scores=read_jsonl(old_dir/'official_scores.jsonl')
    new_scores=read_jsonl(new_dir/'official_scores.jsonl')
    score_rows=old_scores+new_scores
    scoremap={r['score_key']:r for r in score_rows}
    assert len(scoremap)==len(score_rows),'Score key collision or duplicates'
    all_init=old_init+new_init
    assert len({int(r['task_id']) for r in all_init})==len(all_init),'Overlapping development IDs'
    all_fix=old_fix+new_fix
    assert len({(int(r['task_id']),r['condition']) for r in all_fix})==len(all_fix)
    failed={int(r['task_id']):r for r in all_init if r['visible_status']!='passed'}
    lookup={(int(r['task_id']),r['condition']):r for r in all_fix}
    assert set(lookup)=={(tid,c) for tid in failed for c in MODES}
    detail=[]
    for tid,start in sorted(failed.items()):
        entry={'task_id':tid,'first_failure_type':start['visible_status'],'starting_completion_tokens':start['completion_tokens']}
        for c in MODES:
            r=lookup[(tid,c)]
            recovered=any(_pass(scoremap,f'fixed:{c}:{tid}:{a["correction_index"]}') for a in r['attempts'])
            exact,near=_repeat_stats([start['code']]+[a['code'] for a in r['attempts']],
                                     [start['visible_status']]+[a['visible_status'] for a in r['attempts']])
            entry[c+'_pass']=int(recovered)
            entry[c+'_visible_pass']=int(r['recovered_visible'])
            entry[c+'_generated_tokens']=r['total_completion_tokens']
            entry[c+'_prompt_tokens']=r['total_prompt_tokens']
            entry[c+'_executions']=r['num_corrections']
            entry[c+'_repeated_exact']=exact
            entry[c+'_repeated_near']=near
        detail.append(entry)
    with (new_dir/'fixed_state_by_task.csv').open('w',newline='') as f:
        w=csv.DictWriter(f,fieldnames=list(detail[0]) if detail else ['task_id'])
        w.writeheader();w.writerows(detail)
    fixed={}
    for c in MODES:
        fixed[c]={
            'n_failed_states':len(detail),
            'official_success':sum(r[c+'_pass'] for r in detail),
            'official_rate':sum(r[c+'_pass'] for r in detail)/len(detail) if detail else None,
            'visible_success':sum(r[c+'_visible_pass'] for r in detail),
            'mean_generated_tokens':mean(r[c+'_generated_tokens'] for r in detail) if detail else None,
            'mean_prompt_tokens':mean(r[c+'_prompt_tokens'] for r in detail) if detail else None,
            'exact_repeated_unsuccessful':sum(r[c+'_repeated_exact'] for r in detail),
            'near_repeated_unsuccessful':sum(r[c+'_repeated_near'] for r in detail),
        }
    types={}
    for status in sorted({r['first_failure_type'] for r in detail}):
        rows=[r for r in detail if r['first_failure_type']==status]
        types[status]={'n':len(rows),**{c+'_success':sum(r[c+'_pass'] for r in rows) for c in MODES}}
    summary={
        'scope':'DEV ONLY (30 old + new dev), not untouched final evaluation',
        'n_dev_tasks':len(all_init),'n_fixed_failed_states':len(detail),
        'n_visible_first_attempt_pass':sum(r['visible_status']=='passed' for r in all_init),
        'failure_types':types,'fixed_state_results':fixed,
        'paired_C2_minus_C1':_boot([r['c1_pass'] for r in detail],[r['c2_pass'] for r in detail]),
        'paired_C2_minus_C0':_boot([r['c0_pass'] for r in detail],[r['c2_pass'] for r in detail]),
        'fixed_case_records':'fixed_state_by_task.csv',
        'definitions':{
            'recovery':'official base AND plus tests pass after a correction from identical failed C0 state',
            'repeated_exact':'AST-normalized code unchanged from previous attempt AND current visible tests fail',
            'repeated_near':'normalized source similarity >=0.95, nonidentical, AND current visible tests fail',
            'tokens_per_solved':'all generated completion tokens across tasks divided by number of officially solved tasks',
            'three_attempt_success':'ever passed official tests among produced attempts up to k (adaptive visible-test stopping)',
            'confidence_intervals':'paired bootstrap resampling development task IDs; one training seed',
        },
    }
    if include_ablation:
        ab=read_jsonl(new_dir/'no_feedback_c2.jsonl')
        baseline_new=[r for r in detail if int(r['task_id']) in {int(x['task_id']) for x in new_init}]
        by_id={int(r['task_id']):r for r in ab}
        assert set(by_id)=={int(r['task_id']) for r in new_init if r['visible_status']!='passed'}
        yes=[r['c2_pass'] for r in baseline_new]
        no=[int(_pass(scoremap,f'no_feedback:c2:{r["task_id"]}')) for r in baseline_new]
        summary['no_feedback_ablation_new_dev_only']={
            'n_states':len(yes),'with_feedback_successes':sum(yes),
            'without_detailed_feedback_successes':sum(no),
            'paired_difference':_boot(no,yes),
            'interpretation':'Vague retry prompt vs exact original visible-test error; only on newly added development tasks',
        }
    if include_e2e:
        records=read_jsonl(new_dir/'end_to_end.jsonl')
        ids=json.loads((new_dir/'end_to_end_manifest.json').read_text())['task_ids']
        assert {(int(r['task_id']),r['condition']) for r in records}=={(i,c) for i in ids for c in MODES}
        budget={}
        by_task=defaultdict(dict)
        e2e_detail=[]
        for r in records:
            tid,c=int(r['task_id']),r['condition']
            ever=[int(any(_pass(scoremap,f'e2e:{c}:{tid}:{a["attempt"]}') for a in r['attempts'] if a['attempt']<=k)) for k in (1,2,3)]
            exact,near=_repeat_stats([a['code'] for a in r['attempts']], [a['visible_status'] for a in r['attempts']])
            item={'task_id':tid,'condition':c, 'success_1':ever[0],'success_2':ever[1],'success_3':ever[2],
                  'generated_tokens':r['total_completion_tokens'],'prompt_tokens':r['total_prompt_tokens'],
                  'test_executions':len(r['attempts']),'repeated_exact':exact,'repeated_near':near}
            by_task[tid][c]=item;e2e_detail.append(item)
        with (new_dir/'end_to_end_by_task.csv').open('w',newline='') as f:
            w=csv.DictWriter(f,fieldnames=list(e2e_detail[0]) if e2e_detail else ['task_id'])
            w.writeheader();w.writerows(e2e_detail)
        assert all(set(conditions)==set(MODES) for conditions in by_task.values())
        for c in MODES:
            rs=[x for x in e2e_detail if x['condition']==c]
            solved=sum(x['success_3'] for x in rs)
            budget[c]={
                'n_tasks':len(rs), 'success_1':sum(x['success_1'] for x in rs),
                'success_2':sum(x['success_2'] for x in rs),'success_3':solved,
                'rate_1':mean(x['success_1'] for x in rs), 'rate_2':mean(x['success_2'] for x in rs),
                'rate_3':mean(x['success_3'] for x in rs),
                'total_generated_tokens':sum(x['generated_tokens'] for x in rs),
                'tokens_per_solved_problem':sum(x['generated_tokens'] for x in rs)/solved if solved else None,
                'mean_test_executions_per_task':mean(x['test_executions'] for x in rs),
                'exact_repeated_unsuccessful':sum(x['repeated_exact'] for x in rs),
                'near_repeated_unsuccessful':sum(x['repeated_near'] for x in rs),
            }
        summary['end_to_end_budget']=budget
        summary['end_to_end_paired_C2_minus_C1']={
            f'k={k}':_boot([by_task[i]['c1'][f'success_{k}'] for i in ids],
                            [by_task[i]['c2'][f'success_{k}'] for i in ids]) for k in (1,2,3)}
    if include_humaneval:
        h=read_jsonl(new_dir/'humaneval_official_scores.jsonl')
        counts={c:[r['mbppplus_pass'] for r in h if r['condition']==c] for c in MODES}
        m=json.loads((new_dir/'humaneval_manifest.json').read_text())
        assert all(len(counts[c])==len(m['task_ids']) for c in MODES)
        summary['humaneval_plus_single_shot_transfer']={c:{'n':len(counts[c]),'passed':sum(counts[c]),
            'rate':mean(counts[c])} for c in MODES}
        sm={(r['task_id'],r['condition']):r for r in h}
        summary['humaneval_paired_C2_minus_C1']=_boot(
            [int(sm[(tid,'c1')]['mbppplus_pass']) for tid in m['task_ids']],
            [int(sm[(tid,'c2')]['mbppplus_pass']) for tid in m['task_ids']])
    write_json(new_dir/'day5_summary.json',summary)
    return summary


def make_figures(summary,output):
    import matplotlib.pyplot as plt
    import math
    p=Path(output)/'figures';p.mkdir(exist_ok=True)
    fixed=summary['fixed_state_results']
    fig,ax=plt.subplots(figsize=(6.6,4))
    ax.bar(['C0','C1','C2'],[100*fixed[c]['official_rate'] for c in MODES]);ax.set_ylabel('Official MBPP+ recovery (%)')
    ax.set_title(f'Fixed-state recovery (n={summary["n_fixed_failed_states"]} failures)')
    fig.tight_layout();fig.savefig(p/'fixed_recovery.png',dpi=160);plt.close(fig)
    types=summary['failure_types'];cats=list(types)
    if cats:
        fig,ax=plt.subplots(figsize=(7.2,4.3))
        x=list(range(len(cats)));w=.25
        for i,c in enumerate(MODES):
            ax.bar([t+(i-1)*w for t in x], [types[k][c+'_success']/types[k]['n']*100 for k in cats], width=w,label=c.upper())
        ax.set_xticks(x,[f'{k}\n(n={types[k]["n"]})' for k in cats]);ax.set_ylabel('Recovery (%)')
        ax.legend();ax.set_title('Recovery by first-failure type (small groups are exploratory)')
        fig.tight_layout();fig.savefig(p/'by_failure_type.png',dpi=160);plt.close(fig)
    if 'end_to_end_budget' in summary:
        fig,ax=plt.subplots(figsize=(6.6,4))
        for c in MODES:
            b=summary['end_to_end_budget'][c]
            ax.plot([1,2,3],[100*b[f'rate_{k}'] for k in (1,2,3)],marker='o',label=c.upper())
        ax.set_xticks([1,2,3]);ax.set_xlabel('Maximum attempts');ax.set_ylabel('Cumulative official success (%)')
        ax.set_title('End-to-end code generation');ax.legend()
        fig.tight_layout();fig.savefig(p/'attempt_budget.png',dpi=160);plt.close(fig)
        fig,ax=plt.subplots(figsize=(6.6,4))
        vals=[summary['end_to_end_budget'][c]['tokens_per_solved_problem'] for c in MODES]
        ax.bar([c.upper() for c in MODES],[v if v is not None else 0 for v in vals])
        ax.set_ylabel('Total generated tokens / officially solved tasks')
        ax.set_title('Compute efficiency (report alongside solve rate)')
        fig.tight_layout();fig.savefig(p/'tokens_per_solved.png',dpi=160);plt.close(fig)
    return list(sorted(str(x) for x in p.glob('*.png')))

In [ ]:
%%writefile /kaggle/working/day5_score.py
"""Offline CPU-only EvalPlus scoring for Day 5. Run AFTER all model generation.

The scoring process imports hidden tests. NEVER import this in a generation cell.
Executing generated code is dangerous without strong container isolation.
"""
from __future__ import annotations
import argparse,hashlib
from pathlib import Path
from tqdm.auto import tqdm

from day4_eval import append_jsonl, read_jsonl, sha256_bytes
from day4_score import score as score_mbpp


def score_humaneval(input_jsonl, output_jsonl):
    from evalplus.data import get_human_eval_plus,get_human_eval_plus_hash
    from evalplus.evaluate import check_correctness,get_groundtruth
    inp=Path(input_jsonl);out=Path(output_jsonl)
    req=read_jsonl(inp)
    if not req:
        print('No optional HumanEval+ requests: skipping')
        return
    keys=[r['score_key'] for r in req]
    assert len(keys)==len(set(keys)), 'Duplicate scoring keys'
    digest=sha256_bytes(inp.read_bytes());proof=out.with_suffix('.source_sha256')
    if proof.exists() and proof.read_text().strip()!=digest:
        raise ValueError('HumanEval scoring inputs changed since checkpoint creation; do not combine scores.')
    proof.parent.mkdir(parents=True,exist_ok=True);proof.write_text(digest+'\n')
    existing=read_jsonl(out);done={r['score_key']:r for r in existing}
    if len(existing)!=len(done) or not set(done)<=set(keys):
        raise ValueError('HumanEval checkpoint contains conflicting or duplicate keys')
    all_problems=get_human_eval_plus()
    ids={r['task_id'] for r in req}
    assert ids <= set(all_problems), 'Unknown HumanEval+ IDs'
    problems={tid:all_problems[tid] for tid in ids}
    sig=hashlib.sha256((get_human_eval_plus_hash()+'|'+','.join(sorted(ids))).encode()).hexdigest()[:24]
    expected=get_groundtruth(problems,'day5_humaneval_subset_'+sig,[])
    pending=[r for r in req if r['score_key'] not in done]
    with tqdm(total=len(req),initial=len(done),desc='Official HumanEval+ scoring',unit='solution') as bar:
        for r in pending:
            try:
                check=check_correctness('humaneval',0,problems[r['task_id']],r['code'],
                                        expected[r['task_id']],base_only=False,fast_check=True)
            except Exception as exc:
                raise RuntimeError(f'HumanEval infrastructure error on {r["score_key"]}: {exc}') from exc
            base=check['base'][0];plus=check['plus'][0]
            result={'score_key':r['score_key'],'task_id':r['task_id'],'condition':r['condition'],
                    'stage':'humaneval','evalplus_base_status':base,'evalplus_plus_status':plus,
                    'mbppplus_pass':base=='pass' and plus=='pass'}
            append_jsonl(out,result);bar.update();bar.set_postfix(task=r['task_id'],passed=result['mbppplus_pass'])
    print(f'HumanEval+ results saved: {len(read_jsonl(out))}')

if __name__=='__main__':
    ap=argparse.ArgumentParser()
    ap.add_argument('--kind',choices=['mbpp','humaneval'],required=True)
    ap.add_argument('--requests',required=True)
    ap.add_argument('--output',required=True)
    args=ap.parse_args()
    if args.kind=='mbpp':score_mbpp(args.requests,args.output)
    else:score_humaneval(args.requests,args.output)

## D — Restore Day 2, Day 3 and Day 4 inputs

Attach all three saved notebook outputs or ZIP backups using **Add Input**. No Day 1 attachment is necessary. The model-loading helper restores Day 2's split manifest and Day 3's adapters, and our new helper restores Day 4's precomputed development results.

**Do not rerun Day 4.** Those results are reused verbatim.

In [ ]:
import sys
sys.path.insert(0,'/kaggle/working')
from day4_eval import restore_inputs,read_jsonl,write_json,BASE_ID
from day5_eval import restore_day4_results

INPUTS = restore_inputs('/kaggle/input','/kaggle/working/day5_inputs')
OLD, old_manifest = restore_day4_results('/kaggle/input','/kaggle/working/day5_prior_day4')
print('Day 2 manifest:',(INPUTS/'mbpp_split_manifest.json').exists())
for c in ('c1','c2'):
    p=INPUTS/f'adapter_{c}'/'adapter_model.safetensors'
    print(c,'adapter:',p.is_file(),round(p.stat().st_size/1024**2,2),'MB')
print('Day 4 previous task count:',len(old_manifest['task_ids']))
print('Day 4 official scores:',len(read_jsonl(OLD/'official_scores.jsonl')))

### Optional resume: recover Day 5 checkpoint from an attached ZIP

If a previous Day 5 Kaggle session ended, attach `day5_generation_checkpoint.zip` or `gemma_day5_backup.zip` through Add Input. This cell restores saved trajectory checkpoints **before** freezing task lists, without silently overwriting conflicting local files. On the first run, it will simply do nothing.

In [ ]:
from day5_eval import restore_day5_checkpoint
restore_day5_checkpoint('/kaggle/input',OUT)

## E — Freeze new development and future final-holdout task IDs

We exclude all Day 4 development IDs and all fine-tuning IDs. New development tasks are selected without looking at any C0/C1/C2 results. We reserve 70 more MBPP+ test IDs without using them in this notebook. The manifest is saved for reproducibility.

In [ ]:
from day5_eval import freeze_partitions,freeze_e2e_ids
new_problems, partition = freeze_partitions(
    INPUTS, OLD, OUT,
    n_new=NEW_DEV_TASKS, n_final=FINAL_HOLDOUT_TASKS, seed=PARTITION_SEED
)
new_lookup={int(p['task_id']):p for p in new_problems}
e2e_ids=freeze_e2e_ids(new_problems,OUT,n=E2E_TASKS,seed=E2E_SEED) if RUN_E2E else []
print('Old Day 4 dev:',len(partition['prior_day4_dev_ids']))
print('New Day 5 dev:',len(partition['new_dev_ids']))
print('Untouched future final holdout:',len(partition['final_holdout_ids']))
print('End-to-end dev tasks:',len(e2e_ids))
assert not set(partition['new_dev_ids']) & set(partition['final_holdout_ids'])
assert not set(partition['prior_day4_dev_ids']) & set(partition['new_dev_ids'])
assert not partition['training_overlap']
write_json(OUT/'day5_config.json',{
    'model':BASE_ID,'new_dev_n':NEW_DEV_TASKS,'final_reserved_n':FINAL_HOLDOUT_TASKS,
    'e2e_n':E2E_TASKS,'humaneval_n':HUMANEVAL_TASKS,
    'max_new_tokens':MAX_NEW_TOKENS,'max_attempts':MAX_ATTEMPTS,
    'seed_partition':PARTITION_SEED,'seed_e2e':E2E_SEED,'seed_humaneval':HUMANEVAL_SEED,
    'run_e2e':RUN_E2E,'run_no_feedback':RUN_NO_FEEDBACK,'run_humaneval':RUN_HUMANEVAL,
    'generation':'Deterministic do_sample=False, no adapter for C0, same original visible MBPP tests',
})

## F — Load one 4-bit Gemma base and both LoRA adapters

Keep the model loaded through the generation sections; evaluation tests are not imported here. PEFT switches the adapters in-place.

In [ ]:
from day4_eval import load_all_adapters
model, processor = load_all_adapters(INPUTS, BASE_ID)
print('Loaded adapters:',sorted(model.peft_config.keys()))

## G — Generate C0 first attempts on 80 new development tasks

**Smoke test:** first two tasks, then automatically resume over all 80. Each task result is immediately appended to a checkpoint file. `tqdm` tracks progress.

In [ ]:
from day4_eval import record_base_initials
print('Smoke test: 2 new development tasks')
record_base_initials(new_problems[:2],model,processor,OUT,MAX_NEW_TOKENS)
print('Full new development set (resumes previous 2)')
initials = record_base_initials(new_problems,model,processor,OUT,MAX_NEW_TOKENS)
print('Saved initial attempts:',len(initials),'/',len(new_problems))
print('Initial failure distribution:',__import__('collections').Counter(r['visible_status'] for r in initials.values()))

## H — Paired recovery on the same failed C0 states

C0/C1/C2 receive identical failed C0 code and identical visible-test feedback for each failed state. One correction each, with official hidden tests applied only in the later scoring process.

In [ ]:
from day4_eval import record_fixed_corrections
fixed = record_fixed_corrections(
    new_problems,initials,model,processor,OUT,
    max_new_tokens=MAX_NEW_TOKENS,max_corrections=1
)
expected=len([x for x in initials.values() if x['visible_status']!='passed'])*3
assert len(fixed)==expected,(len(fixed),expected)
print('New fixed-state condition records:',len(fixed),'/',expected)

## Optional checkpoint backup after H

If Kaggle stalls later, download this partial ZIP. It includes your completed initial and fixed-state records. It is NOT the final Day 5 backup.

In [ ]:
from zipfile import ZipFile, ZIP_DEFLATED
from IPython.display import FileLink, display
partial=Path('/kaggle/working/day5_generation_checkpoint.zip')
with ZipFile(partial,'w',ZIP_DEFLATED) as z:
    for f in sorted(OUT.rglob('*')):
        if f.is_file():z.write(f,arcname=str(f.relative_to('/kaggle/working')))
with ZipFile(partial) as z:assert z.testzip() is None
print('Checkpoint created:',partial)
display(FileLink(str(partial)))

## I — End-to-end solving with 1, 2, and 3 attempts (RQ3/RQ4)

On 30 pre-selected new development tasks, each model starts from only the problem statement. It retries **only after visible failures**, and stops after a visible pass or attempt 3. Later, hidden-test scoring provides cumulative success within k attempts. All task/condition pairs checkpoint automatically.

In [ ]:
from day5_eval import record_e2e
if RUN_E2E:
    e2e_problems=[new_lookup[i] for i in e2e_ids]
    e2e=record_e2e(e2e_problems,model,processor,OUT,MAX_NEW_TOKENS,MAX_ATTEMPTS)
    print('End-to-end conditions:',len(e2e),'/',len(e2e_ids)*3)
else:
    print('RUN_E2E=False: skipped')

## J — C2 without detailed execution feedback (ablation)

This isolates whether a **specific test error message** helps C2 recover. Both settings start with the same failed code. The ablation receives only a vague statement that the prior code failed. This is a secondary development analysis, not a new model.

In [ ]:
# J — C2 no-feedback ablation (safe after resume or full Run All)
from day4_eval import read_jsonl
from day5_eval import record_no_feedback

# Rebuild the task-id dictionary from Cell G's checkpoint. This avoids
# accidentally using a list created by any inspection/diagnostic cell.
initial_rows = read_jsonl(OUT / "base_initials.jsonl")
assert len(initial_rows) == len(new_problems), (
    f"Expected {len(new_problems)} initial states, found {len(initial_rows)}"
)
initials = {int(row["task_id"]): row for row in initial_rows}
assert len(initials) == len(initial_rows), "Duplicate task IDs in initial checkpoint"
print("Initial states ready for J:", len(initials))

if RUN_NO_FEEDBACK:
    ab = record_no_feedback(
        new_problems, initials, model, processor, OUT, MAX_NEW_TOKENS
    )
    print("No-feedback correction records:", len(ab))
else:
    print("RUN_NO_FEEDBACK=False: skipped")

## K — Optional HumanEval+ transfer: first attempt only

Use **original public HumanEval prompts**, *not hidden EvalPlus tests*, to generate code from all three conditions. This is an exploratory transfer test (not iterative repair). HumanEval+ official scores are computed separately later.

In [ ]:
from day5_eval import freeze_humaneval,record_humaneval
if RUN_HUMANEVAL:
    human_problems=freeze_humaneval(OUT,n=HUMANEVAL_TASKS,seed=HUMANEVAL_SEED)
    human=record_humaneval(human_problems,model,processor,OUT,MAX_NEW_TOKENS)
    print('HumanEval conditions:',len(human),'/',len(human_problems)*3)
else:
    print('RUN_HUMANEVAL=False: skipped')

### Optional pre-scoring backup (recommended)

All model-generation outputs should now be complete. Update the checkpoint ZIP and download it **before** the separate offline scorer, so a scorer interruption or Kaggle disconnection does not erase generated solutions.

In [ ]:
from zipfile import ZipFile,ZIP_DEFLATED
from IPython.display import FileLink,display
partial=Path('/kaggle/working/day5_generation_checkpoint.zip')
with ZipFile(partial,'w',ZIP_DEFLATED) as z:
    for f in sorted(OUT.rglob('*')):
        if f.is_file():z.write(f,arcname=str(f.relative_to('/kaggle/working')))
with ZipFile(partial) as z:assert z.testzip() is None
print('Updated generation checkpoint:',partial)
display(FileLink(str(partial)))

## L — Freeze all predictions, release GPU model, run OFFLINE official EvalPlus tests

**Do not run this stage until Sections G–K have completed.** Here, generated code is frozen first. The model is unloaded, then separate CPU processes import official EvalPlus hidden tests. Existing Day 4 scores are reused; we score only newly generated candidates.

Test execution remains potentially unsafe without a strong sandbox. Prefer an isolated container with no credentials or network access. The Kaggle notebook runner is only a convenience, *not* a security guarantee.

In [ ]:
from day5_eval import create_scoring_requests
import subprocess,gc,os,sys
mbpp_requests,human_requests=create_scoring_requests(
    new_problems,OUT,
    include_e2e=RUN_E2E,include_ablation=RUN_NO_FEEDBACK,include_humaneval=RUN_HUMANEVAL
)
print('MBPP+ official requests:',len(read_jsonl(mbpp_requests)))
print('HumanEval+ official requests:',len(read_jsonl(human_requests)))
# Free the large GPU model; official scoring uses only CPU and a separate process.
del model,processor
gc.collect();torch.cuda.empty_cache()
env=os.environ.copy();env['CUDA_VISIBLE_DEVICES']=''
cmd=[sys.executable,'/kaggle/working/day5_score.py','--kind','mbpp',
     '--requests',str(mbpp_requests),'--output',str(OUT/'official_scores.jsonl')]
print('Running official MBPP+ scorer ...',flush=True)
subprocess.run(cmd,check=True,env=env,cwd='/kaggle/working')
if RUN_HUMANEVAL:
    cmd=[sys.executable,'/kaggle/working/day5_score.py','--kind','humaneval',
         '--requests',str(human_requests),'--output',str(OUT/'humaneval_official_scores.jsonl')]
    print('Running official HumanEval+ scorer ...',flush=True)
    subprocess.run(cmd,check=True,env=env,cwd='/kaggle/working')
print('Official scoring complete.')

## M — RQ1–RQ4 statistics, paired confidence intervals, plots

The analyses combine existing Day 4 development scores and new Day 5 development scores. No final-holdout task is evaluated. Each confidence interval resamples **paired task IDs** rather than treating model outputs as independent.

In [ ]:
from day5_eval import analyze,make_figures
summary=analyze(OLD,OUT,include_e2e=RUN_E2E,
                include_ablation=RUN_NO_FEEDBACK,include_humaneval=RUN_HUMANEVAL)
print('DEV tasks:',summary['n_dev_tasks'])
print('Identical failed states:',summary['n_fixed_failed_states'])
for c in ('c0','c1','c2'):
    r=summary['fixed_state_results'][c]
    print(f'{c.upper()}: {r["official_success"]}/{r["n_failed_states"]} = {100*r["official_rate"]:.1f}% official recovery')
print('Paired C2-C1:',json.dumps(summary['paired_C2_minus_C1'],indent=2))
print('First-failure types:',json.dumps(summary['failure_types'],indent=2))
if RUN_E2E:
    print('End-to-end 1/2/3 attempt results:',json.dumps(summary['end_to_end_budget'],indent=2))
if RUN_NO_FEEDBACK:
    print('No-feedback ablation:',json.dumps(summary['no_feedback_ablation_new_dev_only'],indent=2))
if RUN_HUMANEVAL:
    print('HumanEval+ single-shot transfer:',json.dumps(summary['humaneval_plus_single_shot_transfer'],indent=2))
figures=make_figures(summary,OUT)
print('Saved figures:',figures)

## N — Final reproducibility audit and ZIP backup

Check that all evaluation conditions have scores and that the frozen development sets do not overlap. The backup includes **day5 outputs, scripts, plots, and prior Day4 summary**, but not the Day3 LoRA adapter weights. Preserve your Day 3 ZIP separately.

**Download this ZIP before ending your Kaggle session**, then click Save Version → Quick Save, retaining output files if the option appears.

In [ ]:
from zipfile import ZipFile,ZIP_DEFLATED
from IPython.display import FileLink,display

assert summary['n_dev_tasks']==len(partition['prior_day4_dev_ids'])+NEW_DEV_TASKS
assert not set(partition['new_dev_ids']) & set(partition['final_holdout_ids'])
assert not set(partition['prior_day4_dev_ids']) & set(partition['final_holdout_ids'])
assert summary['n_fixed_failed_states'] >= 1
for c in ('c0','c1','c2'):
    assert summary['fixed_state_results'][c]['n_failed_states']==summary['n_fixed_failed_states']

required=[OUT/'day5_summary.json',OUT/'day5_partition_manifest.json',
          OUT/'base_initials.jsonl',OUT/'fixed_corrections.jsonl',OUT/'scoring_inputs.jsonl',
          OUT/'official_scores.jsonl',OLD/'day4_summary.json']
missing=[str(p) for p in required if not p.is_file()]
if missing:raise FileNotFoundError(f'Missing required results: {missing}')
backup=Path('/kaggle/working/gemma_day5_backup.zip')
with ZipFile(backup,'w',ZIP_DEFLATED,compresslevel=6) as z:
    for root in [OUT,OLD]:
        for p in sorted(root.rglob('*')):
            if p.is_file():z.write(p,arcname=str(p.relative_to('/kaggle/working')))
    for name in ['day5_eval.py','day5_score.py','day4_eval.py','day4_score.py','agent_loop.py','test_runner.py']:
        p=Path('/kaggle/working')/name
        z.write(p,arcname='day5_src/'+name)
with ZipFile(backup) as z:
    bad=z.testzip()
    assert bad is None,f'ZIP integrity failed: {bad}'
    print('Backup ZIP verified:',len(z.namelist()),'files')
print('Backup size MB:',round(backup.stat().st_size/1024**2,2))
display(FileLink(str(backup)))

### Research interpretation guide

- **RQ1:** Fixed-state recovery is the cleanest comparison of C1 vs. C2. Report absolute successes, percentages, paired task-level CI, and discordant task cases.
- **RQ2:** Report failure types with their *sample sizes*. Syntax/runtime failures may be too rare for reliable per-category conclusions.
- **RQ3:** Count exact and near-identical unsuccessful edits, executions per problem, and generated tokens per solved problem **alongside solve rate**.
- **RQ4:** End-to-end cumulative success within 1/2/3 attempts uses the same tasks for all conditions; HumanEval+ is a **single-shot exploratory** transfer test, not repository-level generalization.

**Do not treat a lower training loss or a nonzero point estimate as proof of improved recovery.** We have one fine-tuning seed, a small recovery dataset, and a development-only sample. Final untouched-test evaluation and limitations belong in the paper.